# 12 — Legacy ISIC preprocessing inputs only

This notebook is retained only to download and preprocess the two ISIC array pairs used as inputs by notebook 13. **Do not run its final build cell.** The old HAM10000+ISIC concatenation is disabled because it used fabricated ISIC lesion IDs and could duplicate HAM10000. After cells 1–7 finish, run `13_clean_dataset_v2.ipynb`.


In [ ]:
# --- Setup ---
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/zkoymen/melanoma-detection-ham10000.git"
REPO_REF = "codex/clean-data-protocol"
CANDIDATE_PATHS = [
    Path.cwd(),
    Path("/content/melanoma-detection-ham10000"),
]

project_root = None
for p in CANDIDATE_PATHS:
    if (p / "src").exists() and (p / "config.py").exists():
        project_root = p
        break

if project_root is None:
    project_root = Path("/content/melanoma-detection-ham10000")
    subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(project_root)], check=True)
else:
    subprocess.run(["git", "-C", str(project_root), "pull"], check=True)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import config
config.ensure_drive_dirs()
print("OK — Data dir:", config.DATA_DIR)


In [ ]:
# --- Read the actual stored size of X_all.npy (224 or 448?) ---
import numpy as np
from pathlib import Path

x_path = config.DATA_DIR / "X_all.npy"
X_tmp = np.load(x_path, mmap_mode='r')
ACTUAL_SIZE = X_tmp.shape[1]   # 224 or 448
print(f"X_all.npy shape: {X_tmp.shape}")
print(f"ISIC 2019 will be preprocessed to this size ({ACTUAL_SIZE}px)")

In [ ]:
# --- Kaggle auth ---
from pathlib import Path

kaggle_json = Path("/root/.kaggle/kaggle.json")
if not kaggle_json.exists():
    from google.colab import files
    print("Upload kaggle.json (kaggle.com -> top-right profile -> Settings -> API -> Create New Token)")
    uploaded = files.upload()
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    import shutil
    shutil.move(list(uploaded.keys())[0], str(kaggle_json))
    kaggle_json.chmod(0o600)
    print("Done.")
else:
    print("kaggle.json already present.")

In [ ]:
# --- Download ISIC 2019 (~10 GB, ~15 min) ---
import subprocess
from pathlib import Path

ISIC_DIR = Path("/content/isic2019")
ISIC_DIR.mkdir(exist_ok=True)

csv_path = ISIC_DIR / "ISIC_2019_Training_GroundTruth.csv"
if not csv_path.exists():
    print("Downloading (~10 GB, ~15 min)...")
    result = subprocess.run(
        ["kaggle", "datasets", "download",
         "-d", "andrewmvd/isic-2019",
         "-p", str(ISIC_DIR), "--unzip"],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(result.stderr[-2000:])
        raise RuntimeError("Download failed - is kaggle.json correct?")
    print("Download complete.")
else:
    print("Already downloaded.")

print("Files:", [f.name for f in sorted(ISIC_DIR.iterdir())[:10]])

In [ ]:
# --- MEL list + stratified NON-MEL sample (same count as MEL) ---
import pandas as pd, os, numpy as np
from pathlib import Path

gt = pd.read_csv(csv_path)
# ISIC 2019 GT columns: image, MEL, NV, BCC, AK, BKL, DF, VASC, SCC, UNK
NONMEL_COLS = [c for c in ["NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"] if c in gt.columns]

mel_ids = gt[gt["MEL"] == 1]["image"].tolist()
print(f"Total MEL: {len(mel_ids)}")

# Locate the image folder
IMG_DIR = None
for candidate in [ISIC_DIR / "ISIC_2019_Training_Input", ISIC_DIR / "train", ISIC_DIR]:
    if candidate.exists() and any(candidate.glob("*.jpg")):
        IMG_DIR = candidate
        break
    for sub in candidate.glob("*/"):
        if any(sub.glob("*.jpg")):
            IMG_DIR = sub
            break
    if IMG_DIR:
        break
if IMG_DIR is None:
    for root, dirs, files in os.walk(str(ISIC_DIR)):
        if any(f.endswith(".jpg") for f in files):
            IMG_DIR = Path(root)
            break
print("Image folder:", IMG_DIR)

# Stratified-sample NON-MEL by class (to match the MEL count)
n_target = len(mel_ids)
nonmel_df = gt[gt["MEL"] == 0].copy()

def _row_class(r):
    for c in NONMEL_COLS:
        if r[c] == 1:
            return c
    return "OTHER"

nonmel_df["cls"] = nonmel_df.apply(_row_class, axis=1)
nonmel_df = nonmel_df[nonmel_df["cls"] != "OTHER"]

picked = []
for c, grp in nonmel_df.groupby("cls"):
    share = int(round(n_target * len(grp) / len(nonmel_df)))
    picked += grp.sample(n=min(share, len(grp)), random_state=42)["image"].tolist()

rng = np.random.default_rng(42)
picked = list(dict.fromkeys(picked))                      # uniq
if len(picked) > n_target:
    picked = list(rng.permutation(picked))[:n_target]
elif len(picked) < n_target:
    extra = [i for i in nonmel_df["image"].tolist() if i not in set(picked)]
    picked += list(rng.permutation(extra))[:(n_target - len(picked))]
nonmel_ids = picked
print(f"Selected NON-MEL (stratified): {len(nonmel_ids)}")

mel_paths    = [(i, IMG_DIR / f"{i}.jpg") for i in mel_ids    if (IMG_DIR / f"{i}.jpg").exists()]
nonmel_paths = [(i, IMG_DIR / f"{i}.jpg") for i in nonmel_ids if (IMG_DIR / f"{i}.jpg").exists()]
print(f"MEL found: {len(mel_paths)}/{len(mel_ids)}  |  NON-MEL found: {len(nonmel_paths)}/{len(nonmel_ids)}")

In [ ]:
# --- Parallel preprocessing: MEL + NON-MEL (4 threads, ~10-12 min) ---
import cv2, numpy as np, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from src.preprocessing import preprocess_for_storage

N_WORKERS = 4
SIZE = ACTUAL_SIZE   # same size as HAM10000 (224)

def _preprocess_list(pairs, tag):
    X = np.zeros((len(pairs), SIZE, SIZE, 3), dtype=np.uint8)
    ids = np.empty(len(pairs), dtype=object)

    def work(args):
        idx, img_id, img_path = args
        img = cv2.imread(str(img_path))
        if img is None:
            return idx, img_id, np.zeros((SIZE, SIZE, 3), np.uint8)
        try:
            rgb, _ = preprocess_for_storage(img, size=SIZE)
            return idx, img_id, rgb
        except Exception:
            return idx, img_id, np.zeros((SIZE, SIZE, 3), np.uint8)

    tasks = [(i, img_id, p) for i, (img_id, p) in enumerate(pairs)]
    t0, n_done = time.time(), 0
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        futs = {pool.submit(work, t): t[0] for t in tasks}
        for fut in as_completed(futs):
            idx, img_id, rgb = fut.result()
            X[idx], ids[idx] = rgb, img_id
            n_done += 1
            if n_done % 500 == 0:
                el = time.time() - t0
                print(f"  [{tag}] {n_done}/{len(pairs)}  {n_done/el:.1f} img/s")
    print(f"  [{tag}] done: {len(pairs)} images, {(time.time()-t0)/60:.1f} min")
    return X, ids

X_mel, ids_mel = _preprocess_list(mel_paths, "MEL")
X_non, ids_non = _preprocess_list(nonmel_paths, "NON-MEL")
print(f"\nX_mel: {X_mel.shape}  |  X_non: {X_non.shape}")

In [ ]:
# --- Save to Drive (mel + non-mel) ---
import numpy as np

np.save(config.DATA_DIR / "X_isic2019_mel.npy",      X_mel)
np.save(config.DATA_DIR / "ids_isic2019_mel.npy",    ids_mel)
np.save(config.DATA_DIR / "X_isic2019_nonmel.npy",   X_non)
np.save(config.DATA_DIR / "ids_isic2019_nonmel.npy", ids_non)

print("Saved:")
for f in ["X_isic2019_mel.npy", "X_isic2019_nonmel.npy"]:
    print(f"  {f}: {(config.DATA_DIR / f).stat().st_size/1e6:.0f} MB")

# Free RAM (the next cell reloads from disk)
del X_mel, X_non, ids_mel, ids_non
import gc; gc.collect()

In [ ]:
# --- Build the balanced, multi-source dataset + write to Drive ---
import importlib, src.data as _sd
importlib.reload(_sd)   # avoid a stale module cache
from src.data import build_balanced_dataset

build_balanced_dataset(config.DATA_DIR, seed=config.SEED, save=True, verbose=True)

# Verify the written files
need = ["X_combined.npy", "y_combined.npy", "ids_combined.npy", "source_combined.npy",
        "lesion_combined.npy", "idx_train_bal.npy", "idx_val_bal.npy", "idx_test_bal.npy"]
print("\n=== Files written to Drive ===")
ok = True
for f in need:
    p = config.DATA_DIR / f
    if p.exists():
        print(f"  OK    {f}  ({p.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"  MISSING {f}")
        ok = False

print("\n" + ("READY. Now run 04_alexnet.ipynb on an A100 (verification step)."
              if ok else "!!! Some files could not be written - check the errors above."))